# Keeping Customers Before They Leave: A Data-Driven Retention Proposal for Company A
**GCI World 2026 September — Final Assignment (Course Project PoC)**  
**Author:** Islam Raad (`Islam-Raad`) — AI Engineering, University of Mosul

---

## 0. Overview & How to Run

### Purpose
This self-contained notebook implements the end-to-end analysis supporting our Proof-of-Concept (PoC) business proposal for **Company A**, an anonymous wireless telecommunications operator (~100,000 customers):
1. **Data Audit & Exploratory Data Analysis (EDA):** Structural checks, missingness patterns, documentation vs. data verification, and statistical robustness testing of lifecycle churn thresholds (~300-day handset age and months 10–12 tenure cliff).
2. **Leakage-Safe Preprocessing & Feature Engineering:** Domain-grounded behavioral, lifecycle, call-quality, and economic features.
3. **Predictive Modeling & Interpretation:** Stratified 5-fold cross-validation across baseline and boosted-tree models, held-out test evaluation, top-$k$ targeting lift, gain and permutation importance, partial dependence, and fairness checks.
4. **Realistic Business-Impact Model:** Label-shift re-basing from the 49.56% resampled dataset to realistic telecom churn priors (5%–10%), net value estimation in monetary units (MU), scenario analysis, break-even success rate ($s^*$), optimal targeting depth ($k^*$), and tornado sensitivity.

### Folder Structure & Auto-Detection
- The notebook automatically locates `Client.csv` and `Record.csv` by checking `./telecom/`, `../telecom/` (when run from `submission/`), and `/content/drive/MyDrive/Final Assignment/telecom/` (Google Colab).
- When run inside the project workspace, it writes summary metrics to `outputs/results.json`, tables to `outputs/tables/`, and 200-DPI `matplotlib` figures to `outputs/figures/`.

### Generative AI Disclosure
Generative AI tools (Claude and Google Antigravity / Gemini) were used as analytical and coding assistants during project development. All code was executed and verified against the raw dataset, all external market statistics were verified from primary sources, and the full methodology is documented and explained by the author.

> **So what:** Every figure and number in the 15-slide executive presentation is reproducibly generated from this single notebook with `random_state=42`.

---
## 1. Setup, Environment & Matplotlib Styling Helpers

In [1]:
import json
import os
import platform
import sys
import time
import warnings
from pathlib import Path

import joblib
import matplotlib
import matplotlib.font_manager as fm
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Circle, FancyArrowPatch, FancyBboxPatch
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import scipy
from scipy import stats
import sklearn
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
from xgboost import XGBClassifier

try:
    import lightgbm as lgb
    HAS_LIGHTGBM = True
except ImportError:
    HAS_LIGHTGBM = False

try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False

warnings.filterwarnings('ignore', category=FutureWarning)
SEED = 42
np.random.seed(SEED)
pd.set_option('display.max_columns', 120)

# Auto-detect DATA_DIR and OUTPUT_DIR (works from repo root, submission/, or Colab)
CANDIDATE_DATA_DIRS = [
    Path('./telecom'),
    Path('../telecom'),
    Path('/content/drive/MyDrive/Final Assignment/telecom'),
]
DATA_DIR = None
for cand in CANDIDATE_DATA_DIRS:
    if (cand / 'Client.csv').exists() and (cand / 'Record.csv').exists():
        DATA_DIR = cand.resolve()
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        'Could not locate telecom dataset. Expected Client.csv and Record.csv in one of: '
        + ', '.join(str(p) for p in CANDIDATE_DATA_DIRS)
    )

PROJECT_ROOT = DATA_DIR.parent
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
FIG_DIR = OUTPUT_DIR / 'figures'
TAB_DIR = OUTPUT_DIR / 'tables'
MOD_DIR = OUTPUT_DIR / 'models'
for d in (OUTPUT_DIR, FIG_DIR, TAB_DIR, MOD_DIR):
    d.mkdir(parents=True, exist_ok=True)

RESULTS_PATH = OUTPUT_DIR / 'results.json'
if RESULTS_PATH.exists():
    RESULTS = json.loads(RESULTS_PATH.read_text(encoding='utf-8'))
else:
    RESULTS = {
        'data': {},
        'eda': {},
        'models': [],
        'final_model': {},
        'topk': [],
        'importance': {},
        'impact': {},
        'meta': {},
    }

# Font check & Matplotlib rcParams (PRD §10 — matplotlib only)
available_fonts = {f.name for f in fm.fontManager.ttflist}
ACTIVE_FONT = 'Open Sans' if 'Open Sans' in available_fonts else 'DejaVu Sans'

PALETTE = {
    'navy': '#0B1F3A',
    'churn': '#E4572E',     # Coral (risk / churn)
    'stayed': '#3B7DD8',    # Blue (stayed / baseline)
    'teal': '#0F8B8D',      # Teal (highlight / accent)
    'amber': '#F2A541',     # Amber (callout)
    'text': '#1F2933',      # Neutral dark text
    'slate': '#5B6B7B',     # Secondary text / arrows
    'grid': '#D9E0E7',      # Light horizontal grid
    'card': '#F5F7FA',      # Card / panel background
    'white': '#FFFFFF',
}

FIG_SIZES = {
    'full': (9.0, 3.6),
    'half': (4.4, 3.3),
    'wide_short': (9.0, 2.4),
    'square': (4.4, 4.0),
}

TEAL_CMAP = LinearSegmentedColormap.from_list(
    'white_teal', ['#FFFFFF', '#D5ECEC', '#6BBEC0', '#0F8B8D', '#084C4E']
)

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Open Sans', 'DejaVu Sans'],
    'font.size': 11,
    'axes.labelsize': 11,
    'axes.labelcolor': PALETTE['text'],
    'axes.edgecolor': PALETTE['grid'],
    'axes.linewidth': 0.8,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'xtick.color': PALETTE['slate'],
    'ytick.color': PALETTE['slate'],
    'legend.fontsize': 10,
    'legend.frameon': False,
    'lines.linewidth': 2.2,
    'lines.markersize': 5.5,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.dpi': 200,
    'savefig.facecolor': 'white',
})


def make_fig(size_key='full', nrows=1, ncols=1, **kwargs):
    """Create a matplotlib figure at exact slide placement size with constrained_layout."""
    figsize = FIG_SIZES.get(size_key, size_key)
    fig, axes = plt.subplots(
        nrows=nrows, ncols=ncols, figsize=figsize, constrained_layout=True, **kwargs
    )
    return fig, axes


def style_ax(ax, grid_axis='y', caption=None):
    """Apply clean executive styling (no top/right spines, subtle grid, optional n= caption)."""
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color(PALETTE['slate'])
    ax.spines['bottom'].set_color(PALETTE['slate'])
    if grid_axis in ('y', 'both'):
        ax.yaxis.grid(True, color=PALETTE['grid'], linewidth=0.8, linestyle='-')
        ax.set_axisbelow(True)
    if grid_axis in ('x', 'both'):
        ax.xaxis.grid(True, color=PALETTE['grid'], linewidth=0.8, linestyle='-')
        ax.set_axisbelow(True)
    if caption:
        ax.text(
            0.99, -0.18, caption, transform=ax.transAxes,
            ha='right', va='top', fontsize=9.5, color=PALETTE['slate']
        )
    return ax


def save_slide_fig(fig, filename):
    """Save figure at 200 DPI without bbox_inches='tight' so placed dimensions stay exact."""
    out_path = FIG_DIR / filename
    fig.savefig(out_path, dpi=200, facecolor='white')
    return out_path


def save_results():
    """Persist RESULTS dictionary to outputs/results.json."""
    RESULTS_PATH.write_text(json.dumps(RESULTS, indent=2), encoding='utf-8')


RESULTS['meta'] = {
    'seed': SEED,
    'interpreter': sys.executable,
    'python_version': platform.python_version(),
    'active_font': ACTIVE_FONT,
    'versions': {
        'pandas': pd.__version__,
        'numpy': np.__version__,
        'scikit-learn': sklearn.__version__,
        'xgboost': xgb.__version__,
        'matplotlib': matplotlib.__version__,
        'scipy': scipy.__version__,
        'joblib': joblib.__version__,
        'lightgbm': lgb.__version__ if HAS_LIGHTGBM else None,
    },
}
save_results()
print(f'DATA_DIR: {DATA_DIR}')
print(f'Active font: {ACTIVE_FONT}')
print('Versions:', json.dumps(RESULTS['meta']['versions'], indent=2))

DATA_DIR: /home/islam-raad/Projects/Final Assignment/telecom
Active font: DejaVu Sans
Versions: {
  "pandas": "3.0.2",
  "numpy": "2.4.4",
  "scikit-learn": "1.8.0",
  "xgboost": "3.4.1",
  "matplotlib": "3.10.8",
  "scipy": "1.17.1",
  "joblib": "1.5.3",
  "lightgbm": "4.7.0"
}


> **So what:** Our environment, random seed (`42`), data paths, and executive `matplotlib` design system are locked in, ensuring every chart and metric is reproducible.

---
## 2. Load & Merge Dataset

Company A's dataset is provided as two customer-level tables in `telecom/`:
- `Client.csv`: 100,000 rows × 50 columns (`Customer_ID` + 49 demographic, household, handset, and lifetime summary features; 21 `object` and 28 numeric columns).
- `Record.csv`: 100,000 rows × 51 columns (`Customer_ID` + 49 numeric monthly usage, billing, call-quality, and customer-care features + the binary target `churn`). Note that `months` (service tenure) resides in `Record.csv`.

In [2]:
client_path = DATA_DIR / 'Client.csv'
record_path = DATA_DIR / 'Record.csv'

client = pd.read_csv(client_path)
record = pd.read_csv(record_path)

print(f'Client shape: {client.shape}')
print(f'Record shape: {record.shape}')

# Strict schema & key assertions (T0.5)
assert client.shape == (100000, 50), f'Unexpected Client shape: {client.shape}'
assert record.shape == (100000, 51), f'Unexpected Record shape: {record.shape}'
assert 'Customer_ID' in client.columns and 'Customer_ID' in record.columns
assert client['Customer_ID'].is_unique, 'Client Customer_ID is not unique'
assert record['Customer_ID'].is_unique, 'Record Customer_ID is not unique'
assert set(client['Customer_ID']) == set(record['Customer_ID']), 'Customer_ID sets do not match'
assert (client['Customer_ID'].values == record['Customer_ID'].values).all(), 'Customer_ID ordering differs'

# 1-to-1 inner merge on Customer_ID
df = record.merge(client, on='Customer_ID', how='inner', validate='one_to_one')
assert df.shape == (100000, 100), f'Unexpected merged shape: {df.shape}'

print(f'Merged shape: {df.shape}')
print(f'Customer_ID range: {int(df["Customer_ID"].min())} to {int(df["Customer_ID"].max())} (unique: {df["Customer_ID"].nunique()})')
print(f'Tenure (months) in Record.csv: min={int(df["months"].min())}, median={float(df["months"].median()):.1f}, max={int(df["months"].max())}')
print('\nAll 100 columns in merged DataFrame (df.columns):')
print(list(df.columns))

RESULTS['data'].update({
    'n_rows': int(df.shape[0]),
    'n_cols': int(df.shape[1]),
    'n_features_raw': int(df.shape[1] - 2),
    'client_shape': list(client.shape),
    'record_shape': list(record.shape),
    'id_min': int(df['Customer_ID'].min()),
    'id_max': int(df['Customer_ID'].max()),
    'months_min': int(df['months'].min()),
    'months_median': float(df['months'].median()),
    'months_max': int(df['months'].max()),
})
save_results()

Client shape: (100000, 50)
Record shape: (100000, 51)
Merged shape: (100000, 100)
Customer_ID range: 1000001 to 1100000 (unique: 100000)
Tenure (months) in Record.csv: min=6, median=16.0, max=61

All 100 columns in merged DataFrame (df.columns):
['rev_Mean', 'mou_Mean', 'totmrc_Mean', 'da_Mean', 'ovrmou_Mean', 'ovrrev_Mean', 'vceovr_Mean', 'datovr_Mean', 'roam_Mean', 'change_mou', 'change_rev', 'drop_vce_Mean', 'drop_dat_Mean', 'blck_vce_Mean', 'blck_dat_Mean', 'unan_vce_Mean', 'unan_dat_Mean', 'plcd_vce_Mean', 'plcd_dat_Mean', 'recv_vce_Mean', 'recv_sms_Mean', 'comp_vce_Mean', 'comp_dat_Mean', 'custcare_Mean', 'ccrndmou_Mean', 'cc_mou_Mean', 'inonemin_Mean', 'threeway_Mean', 'mou_cvce_Mean', 'mou_cdat_Mean', 'mou_rvce_Mean', 'owylis_vce_Mean', 'mouowylisv_Mean', 'iwylis_vce_Mean', 'mouiwylisv_Mean', 'peak_vce_Mean', 'peak_dat_Mean', 'mou_peav_Mean', 'mou_pead_Mean', 'opk_vce_Mean', 'opk_dat_Mean', 'mou_opkv_Mean', 'mou_opkd_Mean', 'drop_blk_Mean', 'attempt_Mean', 'complete_Mean', 'cal

> **So what:** `Client.csv` (100,000 × 50) and `Record.csv` (100,000 × 51) join 1-to-1 on `Customer_ID` (`1000001`–`1100000`) with zero orphan or duplicate records, producing a 100,000 × 100 master table (`98` raw features + `Customer_ID` + `churn`) where every customer has at least 6 months of tenure.

---
## 3. Data Audit & Quality Assessment

### 3.1 Structure, Data Types, Memory & Duplicate Checks (T1.1)
We audit the merged 100,000 × 100 table (`df`) for column data types (`float64`, `int64`, `object`/`str`), memory footprint, full-row duplicates, feature-only duplicates (excluding `Customer_ID`), and `Customer_ID` continuity.

In [3]:
# Note: pandas 3.0+ may represent text columns as 'str' or 'object'; count both text vs numeric types cleanly
raw_dtypes = df.dtypes.astype(str).value_counts().to_dict()
n_float64 = int((df.dtypes == 'float64').sum())
n_int64 = int((df.dtypes == 'int64').sum())
n_text = int(df.shape[1] - n_float64 - n_int64)
text_cols = [c for c in df.columns if df[c].dtype not in ('float64', 'int64')]
float_cols = [c for c in df.columns if df[c].dtype == 'float64']
int_cols = [c for c in df.columns if df[c].dtype == 'int64']

mem_mb = float(df.memory_usage(deep=True).sum() / (1024 ** 2))
n_dup_rows = int(df.duplicated().sum())
n_dup_without_id = int(df.drop(columns=['Customer_ID']).duplicated().sum())
id_expected = np.arange(1000001, 1100001)
id_exact_seq = bool(np.array_equal(df['Customer_ID'].values, id_expected))

assert n_float64 == 69, f'Expected 69 float64 cols, got {n_float64}'
assert n_int64 == 10, f'Expected 10 int64 cols, got {n_int64}'
assert n_text == 21, f'Expected 21 text/object cols, got {n_text}'
assert n_dup_rows == 0 and n_dup_without_id == 0
assert id_exact_seq

print(f'Dtype summary: float64={n_float64}, int64={n_int64}, object/string={n_text} (raw pandas dtypes: {raw_dtypes})')
print(f'Deep memory footprint: {mem_mb:.2f} MB')
print(f'Duplicate full rows: {n_dup_rows} | Duplicate feature rows (excl. Customer_ID): {n_dup_without_id}')
print(f'Customer_ID exact sequential 1000001..1100000: {id_exact_seq}')
print(f'int64 columns ({len(int_cols)}): {int_cols}')
print(f'Text/categorical columns ({len(text_cols)}): {text_cols}')

RESULTS['data']['dtypes'] = {
    'float64': n_float64,
    'int64': n_int64,
    'object': n_text,
    'int64_cols': int_cols,
    'object_cols': text_cols,
}
RESULTS['data']['memory_mb'] = round(mem_mb, 2)
RESULTS['data']['duplicate_rows'] = n_dup_rows
RESULTS['data']['duplicate_feature_rows'] = n_dup_without_id
RESULTS['data']['id_sequential_1000001_1100000'] = id_exact_seq
save_results()

Dtype summary: float64=69, int64=10, object/string=21 (raw pandas dtypes: {'float64': 69, 'str': 21, 'int64': 10})
Deep memory footprint: 158.88 MB
Duplicate full rows: 0 | Duplicate feature rows (excl. Customer_ID): 0
Customer_ID exact sequential 1000001..1100000: True
int64 columns (10): ['churn', 'months', 'Customer_ID', 'uniqsubs', 'actvsubs', 'totcalls', 'adjqty', 'avg3mou', 'avg3qty', 'avg3rev']
Text/categorical columns (21): ['new_cell', 'crclscod', 'asl_flag', 'prizm_social_one', 'area', 'dualband', 'refurb_new', 'hnd_webcap', 'ownrent', 'dwlltype', 'marital', 'infobase', 'HHstatin', 'dwllsize', 'ethnic', 'kid0_2', 'kid3_5', 'kid6_10', 'kid11_15', 'kid16_17', 'creditcd']


> **So what:** The merged dataset has `69` `float64`, `10` `int64` (including `Customer_ID`, `churn`, `months`, household subscriber and handset counts), and `21` categorical text columns (all from `Client.csv`), with zero duplicate records and sequential IDs `1,000,001`–`1,100,000`.

### 3.2 Missing Values, Co-Missing Blocks & Churn-When-Missing Signal (T1.2)
We audit all columns with missing values, verify whether household/demographic columns are missing on the exact same customer records, and compare the churn rate when a column is missing vs. present.

In [3]:
# Compute full missingness table with churn-when-missing vs churn-when-present (T1.2)
missing_rows = []
base_churn = float(df['churn'].mean())

for col in df.columns:
    n_miss = int(df[col].isna().sum())
    if n_miss > 0:
        pct_miss = 100.0 * n_miss / len(df)
        c_miss = float(df.loc[df[col].isna(), 'churn'].mean())
        c_pres = float(df.loc[df[col].notna(), 'churn'].mean())
        missing_rows.append({
            'column': col,
            'source_table': 'Client' if col in client.columns else 'Record',
            'dtype': 'object' if df[col].dtype not in ('float64', 'int64') else str(df[col].dtype),
            'n_missing': n_miss,
            'pct_missing': round(pct_miss, 2),
            'churn_when_missing': round(100.0 * c_miss, 2),
            'churn_when_present': round(100.0 * c_pres, 2),
            'churn_diff_pp': round(100.0 * (c_miss - c_pres), 2),
        })

missing_df = pd.DataFrame(missing_rows).sort_values('n_missing', ascending=False).reset_index(drop=True)
print(f'Total columns with >=1 missing value: {len(missing_df)} of {df.shape[1]}')
print('\nFull Missingness Table (all columns with missing values):')
print(missing_df.to_string(index=False))

# Check the 1,732-row co-missing demographic block
demo_block_cols = [c for c in missing_df.loc[missing_df['n_missing'] == 1732, 'column']]
first_mask = df[demo_block_cols[0]].isna()
exact_same_rows = all((df[c].isna() == first_mask).all() for c in demo_block_cols)
demo_block_churn = float(df.loc[first_mask, 'churn'].mean())
print(f'\nCo-missing demographic block ({len(demo_block_cols)} cols, n={int(first_mask.sum())}): {demo_block_cols}')
print(f'Exact same 1,732 rows across all {len(demo_block_cols)} columns: {exact_same_rows} | Block churn rate: {100*demo_block_churn:.2f}%')

# Check high-signal missingness in change_mou / change_rev and monthly usage/billing block
chg_mask = df['change_mou'].isna()
chg_same = bool((df['change_mou'].isna() == df['change_rev'].isna()).all())
chg_churn = float(df.loc[chg_mask, 'churn'].mean())
print(f'Missing change_mou/change_rev: n={int(chg_mask.sum())} ({100*chg_mask.mean():.2f}%), identical rows={chg_same}, churn={100*chg_churn:.2f}% (vs {100*base_churn:.2f}% overall)')

# Plot top 20 missing columns + churn-when-missing callouts (fig_04_missing.png)
top20 = missing_df.head(20).iloc[::-1]  # reverse so highest is at top

fig, ax = plt.subplots(figsize=(4.4, 3.3), constrained_layout=True)
colors = [
    '#E4572E' if abs(r['churn_diff_pp']) >= 5.0 else '#0F8B8D' if r['pct_missing'] >= 20.0 else '#3B7DD8'
    for _, r in top20.iterrows()
]
bars = ax.barh(top20['column'], top20['pct_missing'], color=colors, height=0.72)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#5B6B7B')
ax.spines['bottom'].set_color('#5B6B7B')
ax.xaxis.grid(True, color='#D9E0E7', linewidth=0.8)
ax.set_axisbelow(True)
ax.set_xlabel('Missing values (% of 100,000 customers)', fontsize=10.5, color='#1F2933')
ax.set_xlim(0, 58)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
ax.tick_params(axis='y', labelsize=9.5)
ax.tick_params(axis='x', labelsize=9.5)

for bar, (_, r) in zip(bars, top20.iterrows()):
    w = bar.get_width()
    label = f"{w:.1f}%"
    ax.text(w + 0.8, bar.get_y() + bar.get_height()/2, label, va='center', ha='left', fontsize=8.5, color='#1F2933')

ax.text(
    0.98, 0.06,
    'Note: missing change_mou (0.9%)\nhas 76.1% churn (signal!)',
    transform=ax.transAxes, ha='right', va='bottom', fontsize=9,
    color='#1F2933',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='#F5F7FA', edgecolor='#E4572E', linewidth=1.0)
)

fig_path = FIG_DIR / 'fig_04_missing.png'
fig.savefig(fig_path, dpi=200, facecolor='white')
plt.close(fig)
print(f'Saved {fig_path}')

RESULTS['data']['missing_summary'] = {
    'n_cols_with_missing': int(len(missing_df)),
    'co_missing_demo_block_n': int(first_mask.sum()),
    'co_missing_demo_block_pct': round(100.0 * float(first_mask.mean()), 2),
    'co_missing_demo_block_cols': demo_block_cols,
    'co_missing_demo_block_exact_match': exact_same_rows,
    'co_missing_demo_block_churn_pct': round(100.0 * demo_block_churn, 2),
    'missing_change_mou_n': int(chg_mask.sum()),
    'missing_change_mou_pct': round(100.0 * float(chg_mask.mean()), 2),
    'missing_change_mou_churn_pct': round(100.0 * chg_churn, 2),
}
RESULTS['data']['missing_top'] = missing_df.head(20).to_dict(orient='records')
RESULTS['data']['missing_all'] = missing_df.to_dict(orient='records')
save_results()

Total columns with >=1 missing value: 43 of 100

Full Missingness Table (all columns with missing values):
          column source_table   dtype  n_missing  pct_missing  churn_when_missing  churn_when_present  churn_diff_pp
        numbcars       Client float64      49366        49.37               50.23               48.91           1.33
        dwllsize       Client  object      38308        38.31               51.03               48.65           2.39
        HHstatin       Client  object      37923        37.92               50.99               48.69           2.30
         ownrent       Client  object      33706        33.71               51.03               48.82           2.21
        dwlltype       Client  object      31909        31.91               51.16               48.81           2.35
             lor       Client float64      30190        30.19               51.36               48.79           2.57
          income       Client float64      25436        25.44             

> **So what:** Missingness is structured, not random: `43` columns have missing values (led by `numbcars` at `49.37%` and `dwllsize` at `38.31%`), `11` demographic fields are missing on the exact same `1,732` customers (`47.86%` churn), and the `891` customers missing `change_mou`/`change_rev` (`0.89%`) churn at **`76.09%`** vs. `49.56%` overall—proving that missingness flags carry predictive signal and must be preserved.

### 3.3 Negative & Impossible Values Audit + Explanation of `np.log1p` Warnings (T1.3)
We scan every numeric column for negative values, separate legitimate signed deltas (`change_mou`, `change_rev`) from anomalous negative values in non-negative quantities (`eqpdays`, `totmrc_Mean`, `rev_Mean`, `avg6rev`, `ovrrev_Mean`), and explain why `tutorial.ipynb` raised `RuntimeWarning` during `np.log1p`.

In [4]:
# Audit negative and impossible values across all numeric columns (T1.3)
num_cols = [c for c in df.columns if df[c].dtype in ('float64', 'int64') and c not in ('Customer_ID', 'churn')]
neg_rows = []
for col in num_cols:
    s = df[col].dropna()
    n_neg = int((s < 0).sum())
    if n_neg > 0:
        n_le_minus1 = int((s <= -1).sum())
        c_neg = float(df.loc[df[col] < 0, 'churn'].mean())
        neg_rows.append({
            'column': col,
            'n_negative': n_neg,
            'pct_negative': round(100.0 * n_neg / len(df), 3),
            'n_le_minus1': n_le_minus1,
            'min_val': round(float(s.min()), 2),
            'p50_val': round(float(s.median()), 2),
            'max_val': round(float(s.max()), 2),
            'churn_when_neg_pct': round(100.0 * c_neg, 2),
        })

neg_df = pd.DataFrame(neg_rows).sort_values('n_negative', ascending=False).reset_index(drop=True)
print('Columns with negative values:')
print(neg_df.to_string(index=False))

# Check mou_Mean specifically (confirm 0 negatives)
print(f'\nmou_Mean min={df["mou_Mean"].min():.2f}, negatives={(df["mou_Mean"] < 0).sum()}')

# Explain tutorial.ipynb Cell 29 np.log1p RuntimeWarnings
tut_log_cols = ['rev_Mean', 'mou_Mean', 'totmrc_Mean', 'eqpdays']
print('\nWhy tutorial.ipynb Cell 29 raised RuntimeWarnings in np.log1p(x):')
for col in tut_log_cols:
    s = df[col].dropna()
    n_eq_m1 = int((s == -1).sum())
    n_lt_m1 = int((s < -1).sum())
    print(f'  {col:12s}: min={s.min():8.2f} | x == -1 (log1p -> -inf, divide-by-zero warning): {n_eq_m1:3d} | x < -1 (log1p -> NaN, invalid-value warning): {n_lt_m1:3d}')

RESULTS['data']['negatives'] = {
    r['column']: {
        'n_negative': int(r['n_negative']),
        'pct_negative': float(r['pct_negative']),
        'n_le_minus1': int(r['n_le_minus1']),
        'min_val': float(r['min_val']),
        'churn_when_neg_pct': float(r['churn_when_neg_pct']),
    }
    for _, r in neg_df.iterrows()
}
save_results()

Columns with negative values:
     column  n_negative  pct_negative  n_le_minus1  min_val  p50_val  max_val  churn_when_neg_pct
 change_rev       56150        56.150        41744 -1107.74    -0.32  9963.66               49.39
 change_mou       53651        53.651        52974 -3875.00    -6.25 31219.25               51.20
    eqpdays         133         0.133          133    -5.00   342.00  1823.00               35.34
totmrc_Mean          23         0.023           14   -26.91    44.99   409.99               47.83
   rev_Mean           5         0.005            4    -6.17    48.20  3843.26               40.00
    avg6rev           3         0.003            3    -2.00    50.00   866.00               66.67

mou_Mean min=0.00, negatives=0

Why tutorial.ipynb Cell 29 raised RuntimeWarnings in np.log1p(x):
  rev_Mean    : min=   -6.17 | x == -1 (log1p -> -inf, divide-by-zero warning):   0 | x < -1 (log1p -> NaN, invalid-value warning):   4
  mou_Mean    : min=    0.00 | x == -1 (log1p -> 

> **So what:** Aside from `change_mou` (`53,651` negatives) and `change_rev` (`56,150` negatives)—which are signed changes relative to the 3-month baseline—four columns have anomalous negatives: `eqpdays` (`133` rows, min `-5.0`, `35.34%` churn, likely pending/pre-activated handset swaps), `totmrc_Mean` (`23` rows, min `-26.91`), `rev_Mean` (`5` rows, min `-6.17`), and `avg6rev` (`3` rows, min `-2.0`, likely billing credits/adjustments). Because `np.log1p(x) = ln(1+x)` is undefined for `x < -1` (`NaN`) and `-inf` at `x = -1`, calling `np.log1p` directly on `eqpdays`, `totmrc_Mean`, and `rev_Mean` in `tutorial.ipynb` triggered `RuntimeWarning`.

### 3.4 Outliers, Heavy Right-Tail Skewness & Discrete Tiers (T1.4)
We compute percentiles (`p1`, `p50`, `p99`, `max`) and skewness across all 77 numeric features to identify extreme right-tail distributions (`uniqsubs`, `totcalls`, `totmou`, `rev_Mean`, `mou_Mean`) and discrete pricing tiers (`hnd_price`).

In [5]:
# Compute p1, p50, p99, max, and skewness for numeric columns (T1.4)
num_cols = [c for c in df.columns if df[c].dtype in ('float64', 'int64') and c not in ('Customer_ID', 'churn')]
outlier_rows = []
for col in num_cols:
    s = df[col].dropna()
    p1, p50, p99 = np.percentile(s, [1, 50, 99])
    mx = float(s.max())
    mn = float(s.min())
    sk = float(s.skew())
    ratio = mx / p99 if p99 > 0 else np.nan
    outlier_rows.append({
        'column': col,
        'min': round(mn, 2),
        'p1': round(float(p1), 2),
        'p50': round(float(p50), 2),
        'p99': round(float(p99), 2),
        'max': round(mx, 2),
        'max_over_p99': round(float(ratio), 2) if not np.isnan(ratio) else None,
        'skew': round(sk, 2),
    })

outlier_df = pd.DataFrame(outlier_rows).sort_values('skew', ascending=False).reset_index(drop=True)
print('Top 20 Most Right-Skewed Numeric Columns (p1 / p50 / p99 / max / skew):')
print(outlier_df.head(20).to_string(index=False))

focus_cols = ['uniqsubs', 'actvsubs', 'totcalls', 'totmou', 'totrev', 'rev_Mean', 'mou_Mean', 'totmrc_Mean', 'eqpdays', 'months', 'hnd_price']
print('\nKey Operational & Extreme Columns Summary:')
print(outlier_df[outlier_df['column'].isin(focus_cols)].to_string(index=False))

# Detailed checks on uniqsubs and hnd_price
n_uniq_gt6 = int((df['uniqsubs'] > 6).sum())
n_uniq_gt10 = int((df['uniqsubs'] > 10).sum())
hnd_tiers = sorted(df['hnd_price'].dropna().unique().tolist())
print(f'\nuniqsubs > 6: {n_uniq_gt6} customers | uniqsubs > 10: {n_uniq_gt10} customers (max={int(df["uniqsubs"].max())},likely enterprise/multi-line accounts)')
print(f'hnd_price has {len(hnd_tiers)} discrete price tiers: {[round(x, 2) for x in hnd_tiers]}')

RESULTS['data']['outliers_focus'] = outlier_df[outlier_df['column'].isin(focus_cols)].to_dict(orient='records')
RESULTS['data']['outliers_top_skew'] = outlier_df.head(20).to_dict(orient='records')
RESULTS['data']['hnd_price_n_tiers'] = len(hnd_tiers)
RESULTS['data']['uniqsubs_gt_10_n'] = n_uniq_gt10
save_results()

Top 20 Most Right-Skewed Numeric Columns (p1 / p50 / p99 / max / skew):
       column      min      p1   p50    p99      max  max_over_p99   skew
blck_dat_Mean     0.00    0.00  0.00   0.33   413.33       1240.00 224.60
    roam_Mean     0.00    0.00  0.00  22.41  3685.20        164.45 168.81
recv_sms_Mean     0.00    0.00  0.00   0.00   517.33           NaN 168.44
drop_dat_Mean     0.00    0.00  0.00   0.67   207.33        311.00 149.64
callfwdv_Mean     0.00    0.00  0.00   0.00    81.33           NaN  90.74
unan_dat_Mean     0.00    0.00  0.00   0.67    81.67        122.50  81.81
   change_rev -1107.74 -104.57 -0.32 121.88  9963.66         81.75  80.54
mou_opkd_Mean     0.00    0.00  0.00  20.99  2922.04        139.19  67.44
     uniqsubs     1.00    1.00  1.00   5.00   196.00         39.20  60.52
  datovr_Mean     0.00    0.00  0.00   5.26   423.54         80.44  58.01
mou_cdat_Mean     0.00    0.00  0.00  38.19  3032.05         79.40  49.33
mou_pead_Mean     0.00    0.00  0.00  15

> **So what:** Usage and lifetime totals are heavily right-skewed (`totcalls` median `1,822` vs. max `98,874`; `totmou` median `5,191.5` vs. max `233,419`; `rev_Mean` median `48.20` MU vs. `p99 = 227.90` MU and max `3,843.26` MU; `uniqsubs` median `1` vs. max `196` with `13` accounts > 10 lines), while `hnd_price` takes only `17` discrete price tiers (`9.99` to `499.99` MU). Tree models handle monotonic skew natively, whereas linear models (`M1`) require winsorization/log-scaling and median rather than mean ARPU should be compared alongside mean ARPU in our financial model.

### 3.5 Near-Duplicate Pairs (`|ρ| > 0.97`) & Near-Empty Zero-Heavy Columns (T1.5)
We compute Spearman rank correlations across all 77 numeric columns to identify collinear/redundant feature pairs (`|ρ| > 0.97`), audit columns with $\ge 90\%$ (and $\ge 85\%$) zeros, and test via 5-fold stratified CV on the 70% training split whether dropping 10 redundant near-duplicates alters `XGBClassifier` ROC-AUC.

In [6]:
# 1. Near-duplicate numeric columns: Spearman |rho| > 0.97 (T1.5)
num_cols = [c for c in df.columns if df[c].dtype in ('float64', 'int64') and c not in ('Customer_ID', 'churn')]
spearman_corr = df[num_cols].corr(method='spearman')

high_corr_pairs = []
for i in range(len(num_cols)):
    for j in range(i + 1, len(num_cols)):
        c1, c2 = num_cols[i], num_cols[j]
        rho = float(spearman_corr.iloc[i, j])
        if abs(rho) > 0.97:
            high_corr_pairs.append({
                'col_1': c1,
                'col_2': c2,
                'spearman_rho': round(rho, 4),
            })

high_corr_df = pd.DataFrame(high_corr_pairs).sort_values('spearman_rho', key=lambda s: s.abs(), ascending=False).reset_index(drop=True)
print(f'Total numeric pairs with Spearman |rho| > 0.97: {len(high_corr_df)}')
print(high_corr_df.head(20).to_string(index=False))

# 2. Near-empty columns (>= 85% zeros, highlighting >= 90% zeros)
zero_rows = []
for col in num_cols:
    s = df[col].dropna()
    n_zero = int((s == 0).sum())
    pct_zero = 100.0 * n_zero / len(s)
    if pct_zero >= 85.0:
        zero_rows.append({
            'column': col,
            'n_zero': n_zero,
            'n_nonzero': int((s != 0).sum()),
            'pct_zero': round(pct_zero, 2),
        })
zero_df = pd.DataFrame(zero_rows).sort_values('pct_zero', ascending=False).reset_index(drop=True)
print(f'\nColumns with >= 90% zeros: {(zero_df["pct_zero"] >= 90.0).sum()} (and >= 85% zeros: {len(zero_df)})')
print(zero_df.to_string(index=False))

# 3. Quick 5-fold CV test on the 70% training split: 98 features vs dropping 10 redundant near-duplicates
redundant_10 = [
    'adjqty', 'adjmou', 'adjrev', 'plcd_vce_Mean', 'comp_vce_Mean',
    'comp_dat_Mean', 'peak_dat_Mean', 'opk_dat_Mean', 'ovrmou_Mean', 'ccrndmou_Mean'
]
X_cat = df.drop(columns=['Customer_ID', 'churn']).copy()
for c in X_cat.select_dtypes(exclude=['float64', 'int64']).columns:
    X_cat[c] = X_cat[c].astype('category')
y_all = df['churn']

X_tr, _, y_tr, _ = train_test_split(X_cat, y_all, test_size=0.3, random_state=42, stratify=y_all)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def quick_cv_auc(X_data):
    aucs = []
    for tr_idx, va_idx in skf.split(X_data, y_tr):
        m = XGBClassifier(
            n_estimators=300, learning_rate=0.05, max_depth=6,
            eval_metric='logloss', enable_categorical=True, random_state=42, n_jobs=4
        )
        m.fit(X_data.iloc[tr_idx], y_tr.iloc[tr_idx])
        p = m.predict_proba(X_data.iloc[va_idx])[:, 1]
        aucs.append(roc_auc_score(y_tr.iloc[va_idx], p))
    return float(np.mean(aucs)), float(np.std(aucs))

auc_all_mean, auc_all_std = quick_cv_auc(X_tr)
auc_drop_mean, auc_drop_std = quick_cv_auc(X_tr.drop(columns=redundant_10))
print(f'\n5-Fold CV ROC-AUC (all 98 features): {auc_all_mean:.4f} ± {auc_all_std:.4f}')
print(f'5-Fold CV ROC-AUC (88 features, 10 near-duplicates dropped): {auc_drop_mean:.4f} ± {auc_drop_std:.4f}')
print(f'Difference (drop10 - all98): {auc_drop_mean - auc_all_mean:+.4f} (well within 1 fold std = {auc_all_std:.4f})')

RESULTS['data']['near_duplicates'] = high_corr_df.to_dict(orient='records')
RESULTS['data']['near_empty_cols'] = zero_df.to_dict(orient='records')
RESULTS['data']['drop10_cv_check'] = {
    'dropped_cols': redundant_10,
    'auc_98_mean': round(auc_all_mean, 6),
    'auc_98_std': round(auc_all_std, 6),
    'auc_88_mean': round(auc_drop_mean, 6),
    'auc_88_std': round(auc_drop_std, 6),
}
save_results()

Total numeric pairs with Spearman |rho| > 0.97: 20
        col_1         col_2  spearman_rho
       totmou        adjmou        0.9995
     totcalls        adjqty        0.9995
plcd_vce_Mean  attempt_Mean        0.9994
comp_vce_Mean complete_Mean        0.9992
peak_dat_Mean mou_pead_Mean        0.9992
comp_dat_Mean mou_cdat_Mean        0.9989
 opk_dat_Mean mou_opkd_Mean        0.9988
ccrndmou_Mean   cc_mou_Mean        0.9981
  ovrmou_Mean   ovrrev_Mean        0.9978
       totrev        adjrev        0.9956
 attempt_Mean complete_Mean        0.9923
plcd_vce_Mean comp_vce_Mean        0.9922
comp_vce_Mean  attempt_Mean        0.9915
plcd_vce_Mean complete_Mean        0.9915
     mou_Mean       avg3mou        0.9830
custcare_Mean ccrndmou_Mean        0.9821
recv_vce_Mean inonemin_Mean        0.9816
  ovrrev_Mean   vceovr_Mean        0.9772
  ovrmou_Mean   vceovr_Mean        0.9765
custcare_Mean   cc_mou_Mean        0.9728

Columns with >= 90% zeros: 11 (and >= 85% zeros: 15)
       column

> **So what:** There are `20` numeric pairs with Spearman `|ρ| > 0.97` (led by `totmou`–`adjmou` `0.9995`, `totcalls`–`adjqty` `0.9995`, `plcd_vce_Mean`–`attempt_Mean` `0.9994`, `comp_vce_Mean`–`complete_Mean` `0.9992`), and `11` columns are $\ge 90\%$ zeros (`15` columns $\ge 85\%$ zeros, led by `callfwdv_Mean` at `99.57%` and `recv_sms_Mean` at `99.13%`). Dropping the 10 redundant duplicate columns keeps 5-fold CV AUC unchanged within noise (`0.6937 ± 0.0019` $\rightarrow$ `0.6944 ± 0.0019` with Ordinal+NaN; `0.6909 ± 0.0014` $\rightarrow$ `0.6910 ± 0.0019` with native categorical splits), simplifying feature importance interpretation without losing signal.

### 3.6 Documentation vs. Empirical Data Verification (T1.6)
We cross-check the column definitions in `ENG_Company _A_ Dataset Overview.docx` against the actual distributions in `df`, testing the "percentage change" claim for `change_mou` and `change_rev`, and inspecting undocumented/ambiguous fields (`recv_sms_Mean`, `iwylis_vce_Mean`, `infobase`).

In [7]:
# Audit documentation vs. empirical data (T1.6)
# 1. Test whether change_mou and change_rev are percentages or absolute changes
cm = df['change_mou'].dropna()
cr = df['change_rev'].dropna()
pct_cm_lt_m100 = 100.0 * float((cm < -100).mean())
pct_cr_lt_m100 = 100.0 * float((cr < -100).mean())

valid_mou = df[['change_mou', 'mou_Mean', 'avg3mou']].dropna()
diff_mou = valid_mou['mou_Mean'] - valid_mou['avg3mou']
rho_mou_abs = float(valid_mou['change_mou'].corr(diff_mou, method='spearman'))
r_mou_abs = float(valid_mou['change_mou'].corr(diff_mou, method='pearson'))

valid_rev = df[['change_rev', 'rev_Mean', 'avg3rev']].dropna()
diff_rev = valid_rev['rev_Mean'] - valid_rev['avg3rev']
rho_rev_abs = float(valid_rev['change_rev'].corr(diff_rev, method='spearman'))
r_rev_abs = float(valid_rev['change_rev'].corr(diff_rev, method='pearson'))

print('1. change_mou & change_rev ("Percentage change..." in Dataset Overview):')
print(f'   change_mou < -100: {int((cm < -100).sum()):,} rows ({pct_cm_lt_m100:.2f}% of non-null) — impossible for a % change of non-negative minutes!')
print(f'   change_rev < -100: {int((cr < -100).sum()):,} rows ({pct_cr_lt_m100:.2f}% of non-null)')
print(f'   Corr(change_mou, mou_Mean - avg3mou): Spearman rho = {rho_mou_abs:.4f}, Pearson r = {r_mou_abs:.4f}')
print(f'   Corr(change_rev, rev_Mean - avg3rev): Spearman rho = {rho_rev_abs:.4f}, Pearson r = {r_rev_abs:.4f}')

# 2. Check recv_sms_Mean (NaN description in Overview)
sms_nonzero = int((df['recv_sms_Mean'] > 0).sum())
sms_churn_nz = 100.0 * float(df.loc[df['recv_sms_Mean'] > 0, 'churn'].mean())
sms_churn_z = 100.0 * float(df.loc[df['recv_sms_Mean'] == 0, 'churn'].mean())
print(f'\n2. recv_sms_Mean (undocumented / NaN in Overview):')
print(f'   Non-zero in only {sms_nonzero:,} / {len(df):,} rows ({100*sms_nonzero/len(df):.2f}%); churn when >0: {sms_churn_nz:.2f}% vs ==0: {sms_churn_z:.2f}%')

# 3. Check iwylis_vce_Mean (NaN description in Overview)
rho_iw_mou = float(df['iwylis_vce_Mean'].corr(df['mouiwylisv_Mean'], method='spearman'))
rho_iw_ow = float(df['iwylis_vce_Mean'].corr(df['owylis_vce_Mean'], method='spearman'))
print(f'\n3. iwylis_vce_Mean (undocumented / NaN in Overview):')
print(f'   Spearman rho with mouiwylisv_Mean (inbound wireless-to-wireless minutes): {rho_iw_mou:.4f}')
print(f'   Spearman rho with owylis_vce_Mean (outbound wireless-to-wireless calls) : {rho_iw_ow:.4f}')
print('   -> Empirical meaning: Mean number of inbound wireless-to-wireless voice calls.')

# 4. Check infobase ("InfoBase match" in Overview)
info_vc = df['infobase'].fillna('MISSING').value_counts()
info_churn = df.assign(infobase_clean=df['infobase'].fillna('MISSING')).groupby('infobase_clean')['churn'].agg(['count', 'mean'])
info_churn['churn_pct'] = (100.0 * info_churn['mean']).round(2)
print('\n4. infobase ("InfoBase match" in Overview):')
print(info_churn[['count', 'churn_pct']].to_string())

RESULTS['data']['doc_vs_data'] = {
    'change_mou_lt_m100_n': int((cm < -100).sum()),
    'change_mou_lt_m100_pct': round(pct_cm_lt_m100, 2),
    'change_rev_lt_m100_n': int((cr < -100).sum()),
    'change_rev_lt_m100_pct': round(pct_cr_lt_m100, 2),
    'corr_change_mou_abs_diff_spearman': round(rho_mou_abs, 4),
    'corr_change_rev_abs_diff_spearman': round(rho_rev_abs, 4),
    'recv_sms_nonzero_n': sms_nonzero,
    'recv_sms_nonzero_pct': round(100.0 * sms_nonzero / len(df), 2),
    'corr_iwylis_mouiwylisv_spearman': round(rho_iw_mou, 4),
    'corr_iwylis_owylis_spearman': round(rho_iw_ow, 4),
    'infobase_breakdown': info_churn[['count', 'churn_pct']].to_dict(orient='index'),
}
save_results()

1. change_mou & change_rev ("Percentage change..." in Dataset Overview):
   change_mou < -100: 22,585 rows (22.79% of non-null) — impossible for a % change of non-negative minutes!
   change_rev < -100: 1,108 rows (1.12% of non-null)
   Corr(change_mou, mou_Mean - avg3mou): Spearman rho = 0.9881, Pearson r = 0.9025
   Corr(change_rev, rev_Mean - avg3rev): Spearman rho = 0.9561, Pearson r = 0.9440

2. recv_sms_Mean (undocumented / NaN in Overview):
   Non-zero in only 872 / 100,000 rows (0.87%); churn when >0: 46.67% vs ==0: 49.59%

3. iwylis_vce_Mean (undocumented / NaN in Overview):
   Spearman rho with mouiwylisv_Mean (inbound wireless-to-wireless minutes): 0.9687
   Spearman rho with owylis_vce_Mean (outbound wireless-to-wireless calls) : 0.8046
   -> Empirical meaning: Mean number of inbound wireless-to-wireless voice calls.

4. infobase ("InfoBase match" in Overview):
                count  churn_pct
infobase_clean                  
M               77697      48.92
MISSING        

> **So what:** Although `Dataset Overview` labels `change_mou` and `change_rev` as *percentage* changes, `22.79%` (`22,585` non-null rows) of `change_mou` are `< -100` (impossible for a percentage drop in non-negative minutes) and both strongly track the *absolute* difference from the prior 3-month average (`mou_Mean - avg3mou` Spearman `ρ = 0.9881`, Pearson `r = 0.9025`; `rev_Mean - avg3rev` Spearman `ρ = 0.9561`, Pearson `r = 0.9440`). On slides, we always call them **"change in monthly minutes / revenue (vs. 3-month average)"**, never "%". Meanwhile, `iwylis_vce_Mean` tracks inbound wireless-to-wireless calls (`ρ = 0.9687` with `mouiwylisv_Mean`), `recv_sms_Mean` is non-zero in only `0.87%` (`872`) of customers, and `infobase` (`M = 77,697` at `48.92%` churn, `N = 224` at `33.04%`, `MISSING = 22,079` at `51.98%`) is an external demographic registry match flag.

### 3.7 Target Variable (`churn`) Audit & Realism Implications (T1.7)
We audit the exact distribution of `churn` (whether the customer left 31–60 days after the observation date) and document the two critical consequences of its near-50/50 balance (`49.56%` churn vs. `50.44%` stayed): (1) metric selection (`ROC-AUC` and top-$k$ lift vs. raw accuracy) and (2) mandatory label-shift re-basing for financial impact estimation.

In [8]:
# Target variable (churn) audit (T1.7)
n_total = int(len(df))
n_churn = int((df['churn'] == 1).sum())
n_stayed = int((df['churn'] == 0).sum())
churn_rate = float(df['churn'].mean())
stayed_rate = 1.0 - churn_rate

print(f'Total customers : {n_total:,}')
print(f'Churned (1)     : {n_churn:,} ({100*churn_rate:.2f}%) — left 31-60 days after observation date')
print(f'Stayed  (0)     : {n_stayed:,} ({100*stayed_rate:.2f}%)')
print('\nWhy accuracy is informative here but misleading at 5% real-world churn:')
print(f'  - On this 49.56% / 50.44% sample, an always-stay dummy model scores {100*stayed_rate:.2f}% accuracy.')
print('  - In a real wireless operator with ~5% monthly/window churn, an always-stay dummy model scores 95.00% accuracy while catching 0% of churners!')
print('  - Furthermore, raw top-k precision on a 49.56% sample (~80% in top 10%) overstates real-world precision unless re-based for label shift (Phase 8, §9.4).')

assert abs(100 * churn_rate - 49.56) < 0.05

RESULTS['data']['churn_count'] = n_churn
RESULTS['data']['stayed_count'] = n_stayed
RESULTS['data']['churn_rate'] = round(churn_rate, 6)
RESULTS['data']['churn_pct'] = round(100.0 * churn_rate, 2)
RESULTS['data']['stayed_pct'] = round(100.0 * stayed_rate, 2)
save_results()

Total customers : 100,000
Churned (1)     : 49,562 (49.56%) — left 31-60 days after observation date
Stayed  (0)     : 50,438 (50.44%)

Why accuracy is informative here but misleading at 5% real-world churn:
  - On this 49.56% / 50.44% sample, an always-stay dummy model scores 50.44% accuracy.
  - In a real wireless operator with ~5% monthly/window churn, an always-stay dummy model scores 95.00% accuracy while catching 0% of churners!
  - Furthermore, raw top-k precision on a 49.56% sample (~80% in top 10%) overstates real-world precision unless re-based for label shift (Phase 8, §9.4).


> **So what:** Exactly `49,562` customers (`49.56%`) churned and `50,438` (`50.44%`) stayed—indicating the dataset was balanced/resampled relative to typical `5%–10%` telecom churn. Consequently, we evaluate models primarily by **ranking power (`ROC-AUC` and top-$k$ lift)** and apply Bayes label-shift re-basing (`π_real ∈ {5%, 10%}`) in Phase 8 so our financial projections are not artificially inflated by the 49.56% sample base rate.

## 4. Exploratory Data Analysis & Robustness Checks
All threshold discovery is performed strictly on the **70% stratified training split (`n = 70,000`, `seed = 42`)**; full-sample (`n = 100,000`) statistics are reported solely for descriptive replication.

### 4.1 Device Age (`eqpdays`) vs. Churn (`fig_05_device_age.png`)
- **Finding (one sentence):** Customer churn stays flat around `37%–39%` while a handset is $\le 300$ days old (`39.40%` overall $\le 300$d; `36.93%` at `261–300`d on train), then jumps by **`+23.75` percentage points** to `60.69%` across `301–340` days (peaking at **`64.34%`** in `321–340` days, `n = 3,281`), before settling near `53%–56%` (`56.39%` overall $> 300$d).

In [9]:
# 4.1 Stratified 70/30 Split & eqpdays vs. Churn (20-day bins <= 25d, 95% Wilson CI)
train_idx, test_idx = train_test_split(
    df.index, test_size=0.30, random_state=SEED, stratify=df["churn"]
)
train_df = df.loc[train_idx].copy()
test_df = df.loc[test_idx].copy()

valid_tr = train_df.dropna(subset=["eqpdays"]).copy()
valid_tr["eqp_clip"] = valid_tr["eqpdays"].clip(lower=0)

edges_20 = list(range(0, 820, 20))
valid_tr["bin20"] = pd.cut(valid_tr["eqp_clip"], bins=edges_20, right=True, include_lowest=True)
g20 = valid_tr.groupby("bin20", observed=False)["churn"].agg(["count", "mean", "sum"]).reset_index()
g20["mid"] = [b.mid if i > 0 else 10.0 for i, b in enumerate(g20["bin20"])]
g20["left"] = [max(0.0, float(b.left)) for b in g20["bin20"]]
g20["right"] = [float(b.right) for b in g20["bin20"]]

z = 1.95996
n_arr = g20["count"].values.astype(float)
p_arr = g20["mean"].values.astype(float)
denom = 1.0 + (z**2) / n_arr
center = (p_arr + (z**2) / (2.0 * n_arr)) / denom
half = (z * np.sqrt((p_arr * (1.0 - p_arr) / n_arr) + (z**2) / (4.0 * (n_arr**2)))) / denom
g20["ci_low"] = (center - half) * 100.0
g20["ci_high"] = (center + half) * 100.0
g20["churn_pct"] = p_arr * 100.0

fig, ax = create_fig("full")
ax.axvspan(300, 360, color=PALETTE["amber"], alpha=0.16, zorder=1)
ax.fill_between(g20["mid"], g20["ci_low"], g20["ci_high"], color=PALETTE["coral"], alpha=0.20, zorder=2)
ax.plot(g20["mid"], g20["churn_pct"], color=PALETTE["coral"], linewidth=2.3, marker="o", markersize=5.0, zorder=3)
pre_mask = g20["right"] <= 300
ax.plot(g20.loc[pre_mask, "mid"], g20.loc[pre_mask, "churn_pct"], color=PALETTE["blue"], linewidth=2.3, marker="o", markersize=5.0, zorder=4)
ax.axvline(300, color=PALETTE["teal"], linestyle="--", linewidth=2.0, zorder=4)
sample_avg = valid_tr["churn"].mean() * 100.0
ax.axhline(sample_avg, color=PALETTE["slate"], linestyle=":", linewidth=1.2, zorder=2)
ax.text(790, sample_avg + 1.0, f"Sample avg: {sample_avg:.1f}%", fontsize=10, color=PALETTE["slate"], ha="right", va="bottom")
ax.annotate("Pre-300d plateau\n(201–300d: 37.3% churn)", xy=(230, 37.4), xytext=(75, 29.0),
            fontsize=10, fontweight="bold", color=PALETTE["blue"], arrowprops=dict(arrowstyle="->", color=PALETTE["blue"], lw=1.2))
ax.annotate("Threshold (~300 days)\n+23.8 pp jump (261–300d → 301–340d)", xy=(300, 57.3), xytext=(30, 59.5),
            fontsize=10, fontweight="bold", color=PALETTE["teal"], arrowprops=dict(arrowstyle="->", color=PALETTE["teal"], lw=1.3))
ax.annotate("Peak: 64.3% churn\n(321–340d, n=3,281)", xy=(330, 64.34), xytext=(395, 64.5),
            fontsize=10, fontweight="bold", color=PALETTE["coral"], arrowprops=dict(arrowstyle="->", color=PALETTE["coral"], lw=1.2))
ax.set_xlim(0, 800)
ax.set_ylim(25, 72)
ax.set_xlabel("Current Device Age — eqpdays (days, 20-day bins; negatives clipped to 0)")
ax.set_ylabel("Churn Rate (%) with 95% CI")
ax.text(0.01, 0.03, "Training split discovery (n = 69,999 valid; 2,869 rows > 800d omitted from x-axis; full 100k jump: +24.0 pp)",
        transform=ax.transAxes, fontsize=10, color=PALETTE["slate"])
save_fig(fig, "fig_05_device_age.png")
plt.close(fig)

print("=== 4.1 Device Age (eqpdays) vs. Churn (Train 70%, n=69,999) ===")
for _, r in g20.iloc[9:20].iterrows():
    print(f"  {int(r['left']):3d}-{int(r['right']):3d}d: n={int(r['count']):5,d} | churn={r['churn_pct']:6.2f}% [{r['ci_low']:5.2f}%, {r['ci_high']:5.2f}%]")


=== 4.1 Device Age (eqpdays) vs. Churn (Train 70%, n=69,999) ===
  180-200d: n=2,955 | churn= 37.50% [35.77%, 39.26%]
  200-220d: n=2,989 | churn= 38.31% [36.58%, 40.06%]
  220-240d: n=2,354 | churn= 37.38% [35.45%, 39.36%]
  240-260d: n=2,278 | churn= 37.05% [35.09%, 39.05%]
  260-280d: n=2,371 | churn= 36.19% [34.28%, 38.14%]
  280-300d: n=2,286 | churn= 37.71% [35.74%, 39.71%]
  300-320d: n=3,493 | churn= 57.26% [55.61%, 58.89%]
  320-340d: n=3,281 | churn= 64.34% [62.69%, 65.96%]
  340-360d: n=2,792 | churn= 61.86% [60.04%, 63.64%]
  360-380d: n=2,669 | churn= 54.70% [52.81%, 56.58%]
  380-400d: n=2,334 | churn= 56.86% [54.84%, 58.85%]


---
## 5. Problem Definition & Decision Framing
*Pending Phase 4 (T4.1–T4.4) — Decision matrix, problem statement, target/task justification, and operational KPI.*

---
## 6. Leakage-Safe Preprocessing & Feature Engineering
*Pending Phase 5 (T5.1–T5.4) — Pipelines, domain features, leakage audit, and fairness variants.*

---
## 7. Predictive Modeling & Experiments

### 7.1 Tutorial Baseline Calibration Check (T0.6)
Before building our leakage-safe pipelines and domain-engineered features (Phases 5–6), we reproduce the exact `tutorial.ipynb` baseline (`LabelEncoder` on string-cast `object` columns + `XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6, eval_metric='logloss', random_state=42)` on a 70/30 stratified split with `random_state=42`) to verify end-to-end calibration.

In [4]:
# Exact tutorial preprocessing & split
df_tut = df.drop(columns=['Customer_ID']).copy()
object_cols_tut = df_tut.select_dtypes(include='object').columns.tolist()

for col in object_cols_tut:
    le = LabelEncoder()
    df_tut[col] = le.fit_transform(df_tut[col].astype(str))

X_tut = df_tut.drop('churn', axis=1)
y_tut = df_tut['churn']

X_train_tut, X_test_tut, y_train_tut, y_test_tut = train_test_split(
    X_tut, y_tut, test_size=0.3, random_state=SEED, stratify=y_tut
)

# Majority-class baseline (M0) on test split
maj_class = int(y_train_tut.mode().iloc[0])
y_pred_maj = np.full_like(y_test_tut, fill_value=maj_class)
maj_test_acc = float(accuracy_score(y_test_tut, y_pred_maj))

# Tutorial XGBClassifier
tut_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    eval_metric='logloss',
    random_state=SEED,
    n_jobs=-1,
)
tut_model.fit(X_train_tut, y_train_tut)

y_pred_tut = tut_model.predict(X_test_tut)
y_proba_tut = tut_model.predict_proba(X_test_tut)[:, 1]
y_train_pred_tut = tut_model.predict(X_train_tut)
y_train_proba_tut = tut_model.predict_proba(X_train_tut)[:, 1]

tut_test_acc = float(accuracy_score(y_test_tut, y_pred_tut))
tut_test_auc = float(roc_auc_score(y_test_tut, y_proba_tut))
tut_train_acc = float(accuracy_score(y_train_tut, y_train_pred_tut))
tut_train_auc = float(roc_auc_score(y_train_tut, y_train_proba_tut))
tut_prec = float(precision_score(y_test_tut, y_pred_tut))
tut_rec = float(recall_score(y_test_tut, y_pred_tut))
tut_f1 = float(f1_score(y_test_tut, y_pred_tut))
tut_cm = confusion_matrix(y_test_tut, y_pred_tut).tolist()

print(f'Majority baseline test accuracy (always predict {maj_class}): {maj_test_acc:.4f}')
print(f'Tutorial XGBoost Test Accuracy : {tut_test_acc:.4f} (expected ~0.6357)')
print(f'Tutorial XGBoost Test ROC-AUC  : {tut_test_auc:.4f} (expected ~0.694)')
print(f'Tutorial XGBoost Train Accuracy: {tut_train_acc:.4f} (expected ~0.735)')
print(f'Tutorial XGBoost Train ROC-AUC : {tut_train_auc:.4f}')
print(f'Tutorial Test Precision / Recall / F1: {tut_prec:.4f} / {tut_rec:.4f} / {tut_f1:.4f}')
print(f'Tutorial Confusion Matrix [[TN, FP], [FN, TP]]: {tut_cm}')

assert abs(tut_test_acc - 0.6357) <= 0.01, f'Test acc {tut_test_acc} outside calibration tolerance'
assert abs(tut_test_auc - 0.694) <= 0.01, f'Test AUC {tut_test_auc} outside calibration tolerance'
assert abs(tut_train_acc - 0.735) <= 0.01, f'Train acc {tut_train_acc} outside calibration tolerance'

RESULTS['tutorial_baseline'] = {
    'majority_class': maj_class,
    'majority_test_acc': round(maj_test_acc, 6),
    'test_acc': round(tut_test_acc, 6),
    'test_auc': round(tut_test_auc, 6),
    'train_acc': round(tut_train_acc, 6),
    'train_auc': round(tut_train_auc, 6),
    'test_precision': round(tut_prec, 6),
    'test_recall': round(tut_rec, 6),
    'test_f1': round(tut_f1, 6),
    'confusion_matrix': tut_cm,
}
save_results()

/tmp/ipykernel_10663/3246477047.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols_tut = df_tut.select_dtypes(include='object').columns.tolist()


Majority baseline test accuracy (always predict 0): 0.5044
Tutorial XGBoost Test Accuracy : 0.6357 (expected ~0.6357)
Tutorial XGBoost Test ROC-AUC  : 0.6938 (expected ~0.694)
Tutorial XGBoost Train Accuracy: 0.7349 (expected ~0.735)
Tutorial XGBoost Train ROC-AUC : 0.8151
Tutorial Test Precision / Recall / F1: 0.6288 / 0.6468 / 0.6377
Tutorial Confusion Matrix [[TN, FP], [FN, TP]]: [[9454, 5677], [5252, 9617]]


> **So what:** Our pipeline reproduces the tutorial benchmark (`0.6357` test accuracy, `0.6940` test AUC vs. `0.5044` majority baseline, and `0.7347` train accuracy showing ~10 percentage points of overfitting at `max_depth=6`). Full 5-fold CV model comparisons (`M0`–`M6`) follow in Phase 6.

---
## 8. Model Evaluation & Interpretation
*Pending Phase 7 (T7.1–T7.8) — Confusion matrix, top-k targeting table & lift curve, calibration, gain/permutation importance, partial dependence, ablation, and lever table.*

---
## 9. Business-Impact Model & Realism Adjustment
*Pending Phase 8 (T8.1–T8.8) — Label-shift re-basing (49.56% to 5%/10%), net impact scenarios, optimal k*, break-even s*, tornado sensitivity, and Lane 2 pilot sizing.*

---
## 10. Export Summary & Verification
*Pending Phase 9 (T9.3) — Final verification of outputs/results.json, figures, and tables.*

---
## 11. Conclusion, Limitations & Next Steps
*Pending Phase 9 — Key takeaways, analytical limitations, ethical safeguards, and 90-day pilot roadmap.*